# DistilBERT verification run (for the paper)

Re-trains DistilBERT with the same settings as `Phase3_DistilBERT_Colab.ipynb` and produces the
numbers the paper needs but the repo doesn't have: multi-seed mean ± std, confidence intervals,
accuracy per language (including romanized Hinglish), accuracy by VADER confidence, a train/test
duplicate-leakage check, and a sample for human annotation.

**Runtime → Change runtime type → T4 GPU** before starting. About 30 min per seed.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# If the branch has been merged, drop the -b flag.
!git clone -b claude/distilbert-sentiment-cloud-yfmc3r https://github.com/Rudra-Tiwari-codes/Youtube-Sentiment-Analysis.git
%cd Youtube-Sentiment-Analysis
!pip install -q transformers datasets accelerate scikit-learn pandas tabulate vaderSentiment

## Upload the data
Upload `train.csv`, `val.csv`, `test.csv` from your local `data/processed/` (the same files the
original Colab run used, made by `phase1_data_engineering/05_train_test_split.py`).

In [ ]:
import os, shutil
from google.colab import files
os.makedirs('data/processed', exist_ok=True)
for name in files.upload():
    # Colab renames re-uploads to 'train (1).csv'
    clean = name.split(' (')[0].removesuffix('.csv') + '.csv'
    shutil.move(name, f'data/processed/{clean}')
!ls -la data/processed

## Run
Three seeds is what a paper should report; drop to `--seeds 42` if Colab time is tight.

In [ ]:
!python phase3_transformer_models/06_distilbert_verification.py --seeds 42 43 44

In [ ]:
from IPython.display import Markdown
Markdown(open('results/verification/VERIFICATION_REPORT.md').read())

## Optional: MuRIL on the same split
The repo's current `muril_results.txt` is almost cell-for-cell identical to an older DistilBERT
result, so it should be re-run before the paper mentions MuRIL at all. MuRIL is larger; use batch 16.

In [ ]:
# !python phase3_transformer_models/06_distilbert_verification.py --model google/muril-base-cased \
#     --batch-size 16 --seeds 42 --out-dir results/verification_muril

## Download results
Commit `results/` (without the checkpoints) back to the repo, or paste `VERIFICATION_REPORT.md`
into Claude together with `paper_audit/PAPER_AUDIT.md` when fixing the paper.

In [ ]:
!find results -type d -name checkpoints -prune -exec rm -rf {} +
!zip -qr verification_results.zip results
files.download('verification_results.zip')

## After hand-labelling
Fill the `human_label` column of `results/verification/human_annotation_sample.csv`
(Negative / Neutral / Positive, without looking at `vader_label`), upload it, then:

In [ ]:
# !python phase3_transformer_models/06_distilbert_verification.py \
#     --evaluate-human results/verification/human_annotation_sample.csv